# Session1_Task6 — Time Series Forecasting (ARIMA)

## คำสั่งสำคัญที่ใช้
- `ARIMA(series, order=(p,d,q))` → สร้าง model ทำนาย time series
  - `p` = จำนวน autoregressive lags (ดูย้อนหลังกี่วัน)
  - `d` = order of differencing (แปลง non-stationary → stationary)
  - `q` = จำนวน moving average lags
- `.fit()` → ฝึก model กับข้อมูลจริง
- `.get_forecast(steps=30)` → ทำนาย 30 วันข้างหน้า
- `.predicted_mean` → ดึงค่าทำนายออกมา
- `mean_absolute_error()` → วัดความผิดพลาดเฉลี่ย

In [1]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error

In [2]:
# โหลด cleaned data และรวม daily revenue
s = pd.read_csv('sales_transactions_cleaned.csv')
s['date']    = pd.to_datetime(s['date'])
s['revenue'] = (s['quantity'] * s['price']) - pd.to_numeric(s['discount_amount'], errors='coerce').fillna(0)

# groupby date → รวม revenue รายวัน
# .dt.date → ตัดเวลาออก เหลือแค่วันที่
daily = (s.groupby(s['date'].dt.date)['revenue'].sum()
          .reset_index()
          .rename(columns={'date':'Date','revenue':'Sales'}))
daily['Date'] = pd.to_datetime(daily['Date'])
daily = daily.set_index('Date').sort_index()

print('Daily sales shape:', daily.shape)
print('Date range:', daily.index.min(), 'to', daily.index.max())
display(daily.head())

Daily sales shape: (246, 1)
Date range: 2023-11-30 00:00:00 to 2024-08-01 00:00:00


,Sales
Date,
2023-11-30,62.40
2023-12-01,321.25
2023-12-02,435.00
2023-12-03,282.51
2023-12-04,378.12


In [3]:
# แบ่ง train/test (80/20) เพื่อคำนวณ MAE
# .iloc[] → เลือกแถวตามตำแหน่ง (index number)
split = int(len(daily) * 0.8)
train = daily.iloc[:split]
test  = daily.iloc[split:]

print(f'Train: {len(train)} days | Test: {len(test)} days')

# สร้างและ fit ARIMA model
# order=(1,1,1) → ARIMA(p=1, d=1, q=1) เป็นค่าเริ่มต้นที่ดีสำหรับข้อมูล daily sales
model   = ARIMA(train['Sales'], order=(1, 1, 1))
results = model.fit()  # .fit() → ฝึก model ให้เรียนรู้จาก train data

# ทำนายบน test set เพื่อคำนวณ MAE
# .get_forecast(steps) → ทำนายล่วงหน้า steps วัน
# .predicted_mean      → ดึงค่าทำนายออกมาเป็น Series
test_pred = results.get_forecast(steps=len(test)).predicted_mean

# mean_absolute_error → ค่าเฉลี่ยของ |ค่าจริง - ค่าทำนาย|
mae = mean_absolute_error(test['Sales'], test_pred)
print(f'MAE: {mae:.2f}')

Train: 196 days | Test: 50 days
MAE: 1239.39


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [4]:
# ทำนาย 30 วันข้างหน้า (ใช้ข้อมูลทั้งหมด)
full_model   = ARIMA(daily['Sales'], order=(1, 1, 1))
full_results = full_model.fit()

forecast_30  = full_results.get_forecast(steps=30).predicted_mean

# สร้าง date index สำหรับ 30 วันถัดไป
# pd.date_range() → สร้าง list ของวันที่ต่อเนื่อง
last_date    = daily.index.max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=30, freq='D')

# สร้าง DataFrame และ export
forecast_df = pd.DataFrame({
    'Date'            : future_dates.strftime('%Y-%m-%d'),   # .strftime() → แปลงวันที่เป็น string
    'Predicted_Sales' : forecast_30.values.round(2)
})

display(forecast_df)
print(f'\nMAE of model: {mae:.2f}')

forecast_df.to_csv('Session1_SalesForecast.csv', index=False)
print('✅ Saved Session1_SalesForecast.csv')

# === จุดสังเกต ===
# ✔ forecast_df มี 30 แถว
# ✔ คอลัมน์มีแค่ Date และ Predicted_Sales
# ✔ MAE ควรอยู่ในช่วงที่สมเหตุสมผล (ไม่เป็น 0 และไม่สูงผิดปกติ)
# ✔ Predicted_Sales ไม่ควรติดลบ

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


,Date,Predicted_Sales
0,2024-08-02,2893.30
1,2024-08-03,2995.87
2,2024-08-04,2937.64
3,2024-08-05,2970.70
4,2024-08-06,2951.93
5,2024-08-07,2962.59
6,2024-08-08,2956.54
7,2024-08-09,2959.97
8,2024-08-10,2958.02
9,2024-08-11,2959.13



MAE of model: 1239.39
✅ Saved Session1_SalesForecast.csv
